# Step 11: Qwen judge on qualitative claims, 3 tool orders

Runs Qwen3-4B (4-bit, local GPU) on the 194 qualitative claims in the 100 test traces, three times each with the tool results in a different order. Nothing leaves the machine.

**Runtime:** the Step 8 probe ran Qwen3-4B on this GPU at about 3 tokens per second. 194 claims × 3 orders = 582 calls, so expect roughly 2–3 hours. Cell 2 times a few calls first and prints an estimate.

Outputs go to `reports/step_11/`: `run_traces_qwen.log`, `claims_traces_qwen.csv`, `explanations_traces_qwen.csv`, `claims_to_label_traces_qwen.csv`.

In [ ]:
# 1. Setup: run from the repository root and check the GPU
import os, sys, json, time, subprocess
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import torch
import pandas as pd

print("root:", ROOT)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("free / total GB:", [round(x / 1e9, 2) for x in torch.cuda.mem_get_info()] if torch.cuda.is_available() else "-")

In [ ]:
# 2. Smoke test: judge one case's qualitative claims with one order, time each call, estimate the full run
from src.agent.llm import LocalLLM
from src.agent.judge import extract_claims, judge_qualitative, TOOL_ORDER

llm = LocalLLM("qwen", max_new_tokens=120)
trace = json.loads(Path("data/processed/agent/traces/smollm_untuned/test/test-0013.json").read_text(encoding="utf-8"))
claims = [c for c in extract_claims("test-0013", trace["recommendation"]["explanation"]) if c.kind == "qualitative"]

seconds = []
for claim in claims:
    started = time.time()
    judge_qualitative(claim, trace["evidence"], llm, [TOOL_ORDER])
    seconds.append(time.time() - started)
    print(f"{seconds[-1]:5.1f}s  {claim.verdict:13} {claim.text!r}\n        reason: {claim.detail}")

per_call = sum(seconds) / len(seconds)
print(f"\nmean {per_call:.1f}s per call; full run of 582 calls is about {582 * per_call / 3600:.1f} hours")
print("peak GPU GB:", round(torch.cuda.max_memory_allocated() / 1e9, 2))

In [ ]:
# 3. Free the GPU before the full run, which loads its own copy of the model
del llm
import gc; gc.collect()
torch.cuda.empty_cache()
print("free GB:", round(torch.cuda.mem_get_info()[0] / 1e9, 2))

In [ ]:
# 4. Full run: all 100 traces, Qwen judge, 3 tool orders. Output streams here and to the log.
log_path = ROOT / "reports/step_11/run_traces_qwen.log"
command = [sys.executable, "scripts/run_judge.py", "--source", "traces", "--llm", "qwen", "--orders", "3"]
started = time.time()
with open(log_path, "w", encoding="utf-8") as log, subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace") as process:
    for line in process.stdout:
        print(line, end="")
        log.write(line)
print(f"\nexit code {process.returncode}, {(time.time() - started) / 60:.1f} minutes")

In [ ]:
# 5. Sanity checks on the LLM layer. For inspection only: do not report LLM counts until the hand labels are scored.
claims = pd.read_csv("reports/step_11/claims_traces_qwen.csv")
asked = claims[claims.kind == "qualitative"]
per_order = asked.actual.str.split(";")
print("qualitative claims:", len(asked))
print("verdict in the fixed order:", asked.verdict.value_counts().to_dict())
print("judge_error in any order:", int(per_order.map(lambda v: "judge_error" in v).sum()))
print("verdict changes across orders:", int(per_order.map(lambda v: len(set(v)) > 1).sum()))
pd.set_option("display.max_colwidth", 120)
asked[["case_id", "text", "actual", "detail"]].head(20)

## Next: hand labels

Open `reports/step_11/claims_to_label_traces_qwen.csv`. It has no judge verdict, so labels are blind. For each row, fill `label` with `error`, `ok` or `skip` (cannot tell), using the tool results in that case's trace. Optionally add a `note`.

Then run the cell below.

In [ ]:
# 6. Score the judge against the hand labels (run after labelling)
!"{sys.executable}" scripts/judge_agreement.py --tag traces_qwen